In [ ]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

In [ ]:
!nvidia-smi

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

data_id = "alxxtexxr/VRBI-Dataset"
allow_dir = 'model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2'

data_dir = snapshot_download(
    repo_id=data_id,
    repo_type='dataset',
    allow_patterns=f'{allow_dir}/**',
    local_dir='data',
    local_dir_use_symlinks=False,
)
vision_data_dir = Path(data_dir) / allow_dir

print("Vision data downloaded to:", vision_data_dir)

In [ ]:
import joblib
import numpy as np
from sklearn.cluster import MiniBatchKMeans
import time

# Load all batch file paths and the scaler
batch_paths = [f for f in vision_data_dir.glob('*.npz')]
scaler_path = vision_data_dir / 'scaler.pkl'
scaler = joblib.load(scaler_path)

# Initialize MiniBatchKMeans
k = 16384 # Number of clusters
kmeans = MiniBatchKMeans(
    n_clusters=k,
    init='random',
    batch_size=16384,
    n_init=1,
    reassignment_ratio=0.01,
    max_no_improvement=10,
    random_state=42
)

# Load each batch file one by one and update centroids
# batch_paths -> current: 10 files
for i, batch_path in enumerate(batch_paths):
    print("="*128)
    print(f"Processing batch {i+1}/{len(batch_paths)}: {batch_path}")
    print("="*128)
    
    start_time = time.time()
    
    batch = np.load(batch_path)
    visual_embs = batch['visual_embs'].astype(np.float32) # (N, 2048) -> current: (32_400, 2048)
    visual_embs_norm = scaler.transform(visual_embs) # Normalize using the loaded scaler
    
    print("Normalized visual embeddings shape:", visual_embs_norm.shape)
    print("Mean after normalization (should be close to 0):", visual_embs_norm.mean(axis=0)[:5])
    print("Std after normalization (should be close to 1):", visual_embs_norm.std(axis=0)[:5])
    print()
    
    kmeans.partial_fit(visual_embs_norm)
    
    end_time = time.time()
    iter_time = end_time - start_time
    
    print(f"Time taken: {iter_time:.2f}s")
    print()

In [ ]:
centroids_norm = kmeans.cluster_centers_
centroids = scaler.inverse_transform(centroids_norm)

kmeans_path = vision_data_dir / 'kmeans.pkl'
centroids_norm_path = vision_data_dir / 'centroids_norm.npy'
centroids_path = vision_data_dir / 'centroids.npy'

joblib.dump(kmeans, kmeans_path)
np.save(centroids_norm_path, centroids_norm)
np.save(centroids_path, centroids)

print("K-means model saved to:", kmeans_path)
print("Normalized centroids saved to:", centroids_norm_path)
print("Original centroids saved to:", centroids_path)

In [ ]:
from huggingface_hub import upload_file

upload_file(
    path_or_fileobj=str(kmeans_path),
    path_in_repo=str(kmeans_path.relative_to('/content/data')),
    repo_id='alxxtexxr/VRBI-Dataset',
    repo_type='dataset',
)
upload_file(
    path_or_fileobj=str(centroids_norm_path),
    path_in_repo=str(centroids_norm_path.relative_to('/content/data')),
    repo_id='alxxtexxr/VRBI-Dataset',
    repo_type='dataset',
)
upload_file(
    path_or_fileobj=str(centroids_path),
    path_in_repo=str(centroids_path.relative_to('/content/data')),
    repo_id='alxxtexxr/VRBI-Dataset',
    repo_type='dataset',
)